# EABL Raw Report Schema Decision -- detailed walkthrough

This notebook opens the **raw, unparsed** `DailySalesReportNew` workbook and answers, step by step:

1. How many columns does the raw file actually have?
2. Which columns are *sub-columns* nested under another column?
3. Which columns hold products only?
4. Which column holds the salesperson?
5. Which columns are common across the different sheets in one workbook?
6. How is `quantity` derived from the raw cells?
7. How does `quantity` relate to `product`?
8. A diagram of the product <-> quantity relationship.

...and closes with why the parsed schema keeps only 8 columns:
`source_file, salesperson, customer_code, cust_type, segment, visit_date, product, quantity`.

No helper functions -- every step is plain pandas, read top to bottom.

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

RAW_FILE = "EABL dataset/DailySalesReportNew_0101202615012026.xlsx"

all_sheet_names = pd.ExcelFile(RAW_FILE).sheet_names
print(f"This one workbook has {len(all_sheet_names)} sheets:")
print(all_sheet_names)

raw = pd.read_excel(RAW_FILE, sheet_name=0, header=None)
print()
print("Raw sheet shape (rows, columns):", raw.shape)

## Step 1 -- look at the actual raw sheet

Row 0 is a single free-text title cell (it isn't a normal header at all), and the real headers
don't start until several rows down. Printing it raw shows exactly what a person opening this
file in Excel would see -- including the salesperson's name, buried in plain text.

In [ ]:
title_cell = raw.iloc[0, 1]
print("Row 0, column 1 (the title cell) contains:")
print(repr(title_cell))

## Step 2 -- show the excel file itself, inside the notebook

Here's a real slice of the sheet as Excel laid it out: rows 0-9, columns 24-35. You can see the
title block trailing off, blank rows, then the header rows, then real transaction data starting.

In [ ]:
raw.iloc[0:10, 24:36]

## Step 3 -- total number of columns, and the header rows that explain them

The sheet is **201 columns wide**, but it only has 2 rows worth of *directly usable* labels:

- Row 3 -- product name (or a rollup label like `Achieved`)
- Row 4 -- the field label (`Customer`, `Date`, ...) or the product's unit of measure (`BTL`/`CAS`/`PC`)

A column counts as a **field** only if row 4 has a label and row 3 above it is empty -- otherwise
it's a **product/quantity value** column.

In [ ]:
product_row = raw.iloc[3]
field_row = raw.iloc[4]
data = raw.iloc[5:]

is_value_col = product_row.notna()                      # one column per product
is_field_col = field_row.notna() & product_row.isna()    # a real labeled field

print("TOTAL COLUMNS:", raw.shape[1])
print(f"{is_value_col.sum()} product/quantity columns")
print(f"{is_field_col.sum()} labeled field columns: {field_row[is_field_col].tolist()}")
print(f"{(~is_value_col & ~is_field_col).sum()} empty/spacer columns")

## Step 4 -- which columns are *sub-columns* of another column?

Rows 3-4 aren't the whole story. Two rows further up, **row 2** carries a brand-category label
(`Spirits`, `Beer`, `Alcoholic Pre-Mix Drink`, `Cider/ Perry`, `Returnable`, `Return`) that sits
once at the *start* of a block of product columns -- every product column after it, until the next
label, belongs to it. Those product columns are sub-columns of the row-2 category.

In [ ]:
row2 = raw.iloc[2]
category_labels = row2.dropna()
print("Category labels found in row 2, and the column where each one starts:")
for i, label in category_labels.items():
    print(f"  col {i:>3}  ->  {label}")

Forward-filling row 2 across the columns assigns every product column to its parent category --
turning that one-off label into a real "which group does this product belong to" mapping.

In [ ]:
category = row2.ffill()   # carries each label rightward until the next one appears

product_categories = pd.DataFrame({
    "category": category[is_value_col],
    "product": product_row[is_value_col],
    "unit": field_row[is_value_col],
})

print("How many value-columns fall under each category:")
print(product_categories["category"].value_counts())
print()
print("Sample rows:")
product_categories.head(8)

This also solves a mystery from the column-role check: some product *names* showed up twice.
Now it's clear why -- they appear once as a normal sale under `Returnable`, and again under
`Return`. Same product, two different sub-column groups -- a `sale` vs `return` transaction type
that the current 8-column schema doesn't capture yet.

In [ ]:
dupe_names = product_categories["product"].value_counts()
dupe_names = dupe_names[dupe_names > 1].index

print("Products that appear under more than one category (sale vs return):")
product_categories[product_categories["product"].isin(dupe_names)].sort_values("product")

## Step 5 -- which columns contain products only?

Not every value column is a real product -- `Achieved`, `Targets`, `% Performance` are per-category
rollups (summary numbers), not something a customer bought. Removing those leaves the columns that
hold actual product quantities.

In [ ]:
rollup_labels = {"Achieved", "Targets", "% Performance"}

real_product_cols = [c for c in product_row[is_value_col].index if product_row[c] not in rollup_labels]
rollup_cols = [c for c in product_row[is_value_col].index if product_row[c] in rollup_labels]

print(f"{len(rollup_cols)} rollup columns (not real products): {sorted(set(product_row[c] for c in rollup_cols))}")
print(f"{len(real_product_cols)} columns hold real product quantities")
print(f"{product_row[real_product_cols].nunique()} distinct product NAMES among them "
      f"(fewer than the column count -- Step 4 showed why: sale vs return duplicates)")

## Step 6 -- which column holds the salesperson?

None. There is no `Salesperson` field column anywhere in row 3/4. It only exists in two other
places: the workbook's **sheet tab name**, and buried inside that free-text title cell from Step 1.
The parser uses the sheet name -- this cross-checks it against the title-cell text to prove they
actually agree.

In [ ]:
this_sheet_name = all_sheet_names[0]

match = re.search(r"SalesMan\s*:\s*(.+?)\n", title_cell)
embedded_salesperson = match.group(1).strip() if match else None

print("Sheet tab name:            ", this_sheet_name)
print("Salesperson inside title:  ", embedded_salesperson)
print("Do they agree?             ", this_sheet_name.strip() == embedded_salesperson)

## Step 7 -- which columns are common across the different sheets in this one file?

Every sheet is one salesperson's report, and each salesperson sells a different subset of
products. Do the 7 *field* columns (`Customer`, `Date`, ...) still line up, or only the product
columns differ? Checking a handful of sheets from the same workbook settles it.

In [ ]:
field_sets = {}
shapes = {}
for name in all_sheet_names[:5]:
    df = pd.read_excel(RAW_FILE, sheet_name=name, header=None)
    f_row, p_row = df.iloc[4], df.iloc[3]
    field_sets[name] = tuple(f_row[f_row.notna() & p_row.isna()].tolist())
    shapes[name] = df.shape

for name in all_sheet_names[:5]:
    print(f"{name:32s} shape={shapes[name]}  fields={field_sets[name]}")

print()
print("Same 7 field labels on every sheet checked?", len(set(field_sets.values())) == 1)
print("Same total column count on every sheet checked?", len(set(s[1] for s in shapes.values())) == 1)

**Common across every sheet:** the 7 field columns (`Customer`, `Cust Type`, `Segment`, `Date`,
`Time-In`, `Time-Out`, `TIME AT CALL`) -- always the same labels, in the same order.
**Different per sheet:** the total column count and the exact product columns, because each
salesperson's route carries a different product catalog.

## Step 8 -- how is `quantity` derived?

Under the hood every cell in a product column is stored as **text**, like `'0.00'`, not a number.
`quantity` is derived by converting that text to a float -- `0` means the rep visited but didn't
sell that product on that call; anything else is the amount actually sold.

In [ ]:
sample_cell = data[real_product_cols[0]].iloc[0]
print("Raw cell value:", repr(sample_cell), " Python type:", type(sample_cell).__name__)

def to_qty(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

print("Converted with to_qty():", to_qty(sample_cell))

## Step 9 -- how does `quantity` relate to `product`?

One raw row is one customer visit, with up to ~130 product columns sitting side by side on it.
Most are `0.00` (not sold that visit); a handful are non-zero. `quantity` is simply *whichever
number sits in that row, under that product's column* -- so every non-zero cell becomes one
`(product, quantity)` pair for that visit.

In [ ]:
example_row = data.loc[5]   # a real visit with several products sold
records = []
for c in real_product_cols:
    q = to_qty(example_row[c])
    if q:
        records.append({"customer_code": example_row[2], "visit_date": example_row[11],
                         "product": product_row[c], "quantity": q})

print(f"Row 5 has {len(real_product_cols)} product columns, but only {len(records)} of them are non-zero.")
print("Melted into (product, quantity) pairs, this ONE row becomes these output records:")
pd.DataFrame(records)

That's the melt, in full: run this same logic across every row and every product column, and the
~200-column-wide sheet turns into a long, tidy table with just two columns -- `product` and
`quantity` -- one row per item actually sold.

In [ ]:
records = []
for _, row in data.iterrows():
    for c in real_product_cols:
        q = to_qty(row[c])
        if q:
            records.append({"product": product_row[c], "quantity": q})

melted = pd.DataFrame(records)
print(f"{len(data)} visit-rows x {len(real_product_cols)} product columns "
      f"-> melted into {len(melted)} (product, quantity) records")

## Diagram -- product vs. quantity

The clearest way to see the product <-> quantity relationship is to sum quantity by product and
rank them. This is real data from this one sheet (one salesperson, this reporting period).

In [ ]:
top15 = melted.groupby("product")["quantity"].sum().sort_values(ascending=False).head(15)
top15_sorted = top15.sort_values()   # ascending so the biggest bar lands on top

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(top15_sorted.index, top15_sorted.values, color="#2a78d6", height=0.65)

for bar, value in zip(bars, top15_sorted.values):
    ax.text(bar.get_width() + top15_sorted.max() * 0.01, bar.get_y() + bar.get_height() / 2,
            f"{value:,.1f}", va="center", ha="left", fontsize=9, color="#333333")

ax.set_xlabel("Total quantity sold (this sheet)")
ax.set_title("Top 15 products by quantity -- one salesperson, one sheet", loc="left",
              fontsize=12, fontweight="bold")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, color="#e0e0e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(axis="y", length=0)
plt.tight_layout()
plt.show()

## Step 10 -- full column-by-column breakdown

Steps 3-5 classified columns in pieces. Putting every column -- its category, its label or product
name, its role, and why it's kept or dropped -- into one table gives the complete picture of the
raw file, column by column.

In [ ]:
kept_field_labels = {"Customer", "Cust Type", "Segment", "Date"}

column_breakdown = []
for i in range(raw.shape[1]):
    if is_field_col[i]:
        label = field_row[i]
        role = "FIELD"
        relevance = ("KEPT -- part of the who/what-type/when transaction grain" if label in kept_field_labels
                     else "DROPPED -- operational metadata, outside the transaction grain")
    elif is_value_col[i]:
        if product_row[i] in rollup_labels:
            role = "ROLLUP"
            relevance = "DROPPED -- per-category summary (Achieved/Targets/% Performance), not a real sale"
        else:
            role = "PRODUCT"
            relevance = "KEPT -- melted into 'product' + 'quantity'"
    else:
        role = "EMPTY"
        relevance = "DROPPED -- blank spacer column"

    column_breakdown.append({
        "col": i,
        "category": category[i],
        "name": product_row[i] if pd.notna(product_row[i]) else field_row[i],
        "role": role,
        "relevance": relevance,
    })

column_breakdown = pd.DataFrame(column_breakdown)
pd.set_option("display.max_rows", None)
column_breakdown

In [ ]:
print("How many of the 201 columns fall into each role:")
print(column_breakdown["role"].value_counts())
print()
print("Kept vs dropped:")
print(column_breakdown["relevance"].str.split(" -- ").str[0].value_counts())

## Step 11 -- how much does each customer buy of one brand?

There's no `brand` column -- a brand is just the leading word(s) of the product name (`Tusker Lager`,
`Tusker Lite`, `Tusker Malt`, ...). Matching product names that contain a brand keyword, then grouping
by customer, answers "how much does each customer buy of brand X".

In [ ]:
cust_name_col = field_row[field_row == "Customer"].index[0]
cust_code_col = cust_name_col - 1            # the code sits one column left of the name
customer_code = data[cust_code_col].ffill()  # 'Customer' only prints on the first row of each visit block

sales = []
for idx, row in data.iterrows():
    for c in real_product_cols:
        q = to_qty(row[c])
        if q:
            sales.append({"customer_code": customer_code[idx], "product": product_row[c], "quantity": q})

sales = pd.DataFrame(sales)
sales = sales[sales["customer_code"].astype(str).str.match(r"^KE\d+$")]   # drop footer rows (Total, TARGET, ...)
print(f"{len(sales)} real (customer, product, quantity) purchase records")

In [ ]:
BRAND = "Tusker"   # change this to any brand keyword, e.g. "Guinness", "Smirnoff"

brand_sales = sales[sales["product"].str.contains(BRAND, case=False)]
purchases_by_customer = brand_sales.groupby("customer_code")["quantity"].sum().sort_values(ascending=False)

print(f"Top customers by total '{BRAND}' quantity purchased:")
purchases_by_customer.head(15)

## Conclusion -- the final 8-column schema

Everything above collapses into the same 8 columns as before, now with concrete evidence behind
each decision:

In [ ]:
schema = {
    "source_file":   "provenance -- which raw workbook this row came from (added at the file-loop level, not a column in the sheet)",
    "salesperson":   "provenance -- Step 6: the SHEET NAME, cross-checked against the title cell (not a column in the sheet)",
    "customer_code": "the 'Customer' field column -- part of the who/what-type/when transaction grain",
    "cust_type":     "the 'Cust Type' field column -- part of the who/what-type/when transaction grain",
    "segment":       "the 'Segment' field column -- part of the who/what-type/when transaction grain",
    "visit_date":    "the 'Date' field column -- part of the who/what-type/when transaction grain",
    "product":       f"Step 5: melted from {len(real_product_cols)} real product columns ({product_row[real_product_cols].nunique()} distinct names), rollups removed",
    "quantity":      "Step 8/9: the numeric value converted from each product cell, carried alongside 'product' through the melt",
}
for col, reason in schema.items():
    print(f"{col:15s} {reason}")

print()
print("DROPPED from this schema version:", ["Time-In", "Time-Out", "TIME AT CALL",
                                             "Achieved", "Targets", "% Performance", "Customer (name)"])
print()
print("NOT YET CAPTURED, but visible in the raw structure (candidates for a future schema version):")
print(" - product_category (row 2: Spirits / Beer / Alcoholic Pre-Mix Drink / Cider-Perry / Returnable / Return)")
print(" - transaction_type (sale vs return -- Step 4 showed duplicate product names come from this)")